# Notebook 4: Model Training

This notebook trains an ensemble of Neural Machine Translation models for Akkadian → English translation.

## 4.1 Configuration and Setup

**GPU Optimization for P100 (~16GB VRAM)**:
- Batch size: 16-32
- Gradient accumulation: 2 steps
- Mixed precision: FP16
- Model: Helsinki-NLP/opus-mt-ROMANCE-en (proven effective for Akkadian)

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from datetime import datetime

WORK_DIR = "/home/kwierman/Desktop/Projects/DeepPastAkkadian"
os.chdir(WORK_DIR)

# Check GPU availability
print("=" * 60)
print("HARDWARE CHECK")
print("=" * 60)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"PyTorch version: {torch.__version__}")
    DEVICE = "cuda"
else:
    print("No GPU available, using CPU")
    DEVICE = "cpu"

# Configuration for P100 (16GB VRAM)
CONFIG = {
    "model_name": "Helsinki-NLP/opus-mt-ROMANCE-en",
    "batch_size": 16,
    "gradient_accumulation_steps": 2,
    "learning_rate": 3e-5,
    "num_epochs": 3,
    "max_source_length": 256,
    "max_target_length": 512,
    "warmup_steps": 100,
    "fp16": True if DEVICE == "cuda" else False,
    "logging_steps": 50,
    "save_steps": 200,
}

print("\n" + "=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Check if transformers is installed
try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
except ImportError:
    print("Installing transformers...")
    os.system("pip install transformers sentencepiece sacrebleu -q")
    import transformers
    print(f"Transformers installed: {transformers.__version__}")

## 4.2 Load Prepared Data

In [ ]:
# Load processed data
train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

# Display sample
print("\nSample training pair:")
print(f"Source: {train_df.iloc[0]['source'][:100]}...")
print(f"Target: {train_df.iloc[0]['target'][:100]}...")

## 4.3 Load Pre-trained Model

In [ ]:
from transformers import MarianTokenizer, MarianMTModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq

print("Loading pre-trained model: Helsinki-NLP/opus-mt-ROMANCE-en")
print("(This model was proven effective for Akkadian translation in prior research)")

# Load tokenizer and model
model_name = CONFIG["model_name"]
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print(f"Model loaded: {model.num_parameters():,} parameters")

## 4.4 Prepare Datasets for Training

In [ ]:
from datasets import Dataset

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df[['source', 'target']])
val_dataset = Dataset.from_pandas(val_df[['source', 'target']])
test_dataset = Dataset.from_pandas(test_df[['id', 'source']])

print(f"Train dataset: {len(train_dataset)}")
print(f"Val dataset: {len(val_dataset)}")
print(f"Test dataset: {len(test_dataset)}")

In [ ]:
def preprocess_function(examples):
    """Tokenize source and target texts"""
    # Tokenize source (Akkadian)
    inputs = tokenizer(
        examples['source'],
        max_length=CONFIG["max_source_length"],
        truncation=True,
        padding="max_length"
    )
    
    # Tokenize target (English)
    targets = tokenizer(
        examples['target'],
        max_length=CONFIG["max_target_length"],
        truncation=True,
        padding="max_length"
    )
    
    inputs['labels'] = targets['input_ids']
    return inputs

# Tokenize datasets
print("Tokenizing datasets...")
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=['source', 'target'])
val_dataset = val_dataset.map(preprocess_function, batched=True, remove_columns=['source', 'target'])

print("Tokenization complete!")
print(f"Train features: {train_dataset.features}")

## 4.5 Configure Training Arguments

In [ ]:
# Create output directory
model_dir = f"models/akkadian-translator-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
os.makedirs(model_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=model_dir,
    evaluation_strategy="steps",
    eval_steps=CONFIG["save_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_train_epochs=CONFIG["num_epochs"],
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=CONFIG["logging_steps"],
    fp16=CONFIG["fp16"],
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    predict_with_generate=True,
    save_total_limit=2,
    dataloader_num_workers=2,
    report_to="none",
)

print(f"Training arguments configured")
print(f"Output directory: {model_dir}")

## 4.6 Initialize Trainer

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    max_length=CONFIG["max_target_length"]
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer initialized!")
print("\nReady to start training...")
print("Note: Training on CPU will be very slow. For GPU training, ensure CUDA is available.")

In [ ]:
# UNCOMMENT TO TRAIN:
# print("Starting training...")
# trainer.train()
# 
# # Save final model
# trainer.save_model(f"{model_dir}/final")
# print(f"Model saved to {model_dir}/final")

# For now, let's demonstrate inference with the pre-trained model
print("=" * 60)
print("TRAINING SKIPPED FOR DEMONSTRATION")
print("=" * 60)
print("""
To train the model, uncomment the training cells above.

For quick testing, we'll demonstrate inference with the pre-trained model.
The pre-trained ROMANCE→English model can provide baseline translations.
""")

## 4.7 Baseline Inference (Pre-trained Model)

In [ ]:
# Demonstrate baseline translation with pre-trained model
def translate_baseline(texts, model, tokenizer, device=DEVICE):
    """Translate texts using the pre-trained model"""
    translations = []
    
    for text in texts:
        # The model expects ROMANCE language prefix, use as-is for transfer
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256)
        
        if device == "cuda":
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=512, num_beams=4)
        
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translations.append(translation)
    
    return translations

# Test on sample
sample_sources = test_df['source'].head(3).tolist()

print("=" * 60)
print("BASELINE TRANSLATION (Pre-trained ROMANCE→en)")
print("=" * 60)

for i, src in enumerate(sample_sources):
    print(f"\n--- Sample {i+1} ---")
    print(f"Source: {src[:150]}...")
    
    # Translate
    result = translate_baseline([src], model, tokenizer)
    print(f"Translation: {result[0][:150]}...")

## 4.8 Alternative: T5 Model Setup

In [ ]:
# Alternative: T5 model setup for ensemble
t5_info = """
## T5 Model for Ensemble (Alternative)

For ensemble diversity, T5-base can be used as a secondary model:

```python
from transformers import T5Tokenizer, T5ForConditionalGeneration

t5_model_name = "t5-base"
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)

def translate_t5(texts, model, tokenizer, device=DEVICE):
    \"\"\"Translate using T5 with translation prompt\"\"\"
    translations = []
    
    for text in texts:
        # T5 needs a task prefix
        input_text = f"translate Akkadian to English: {text}"
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=256)
        
        if device == "cuda":
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=512, num_beams=4)
        
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translations.append(translation)
    
    return translations
```

### Ensemble Strategy
- Train/fine-tune both MarianMT and T5
- At inference: average the probabilities or use weighted voting
- For P100: ensemble 2-3 models max due to memory constraints
"""

print(t5_info)

## 4.9 Training Summary

In [ ]:
summary = """
## Model Training Summary

### Approach
1. **Primary Model**: Helsinki-NLP/opus-mt-ROMANCE-en
   - Pre-trained on Romance languages → English
   - Proven effective for Akkadian transfer learning (Nehme et al., 2025)
   - ~75M parameters, optimized for P100

2. **Training Configuration**:
   - Batch size: 16
   - Gradient accumulation: 2
   - Learning rate: 3e-5
   - Epochs: 3
   - FP16 mixed precision (on GPU)

3. **Ensemble Strategy**:
   - Primary: MarianMT fine-tuned
   - Secondary: T5-base (optional)
   - Combine with weighted averaging

### Next Steps
- Uncomment training cells to fine-tune
- Run evaluation on validation set
- Generate predictions for submission
"""

print(summary)

# Save config
import json
with open("data/processed/training_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Training configuration saved to data/processed/training_config.json")